In [0]:
from pyspark.sql.functions import date_format, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType(
    [
        StructField("COD_MUNICIPIO", StringType()),
        StructField("NOM_MUNICIPIO", StringType()),
        StructField("VALOR", StringType())
    ]
)
(
    spark.read
        .option("sep", ";")
        .schema(schema)
        .csv("/Volumes/rfb/transient/transient/ibge")
    .withColumn("DATA_REF", date_format(current_timestamp(), "yyyy-MM"))
    .withColumn("ingested_at", current_timestamp())
    .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("DATA_REF")
        .saveAsTable("rfb.raw.raw_municipio_pib")
)

count = spark.table('rfb.raw.raw_municipio_pib').where('DATA_REF == date_format(current_timestamp(), "yyyy-MM")').count()

if count > 0:
    print("Deleting transient folder.")
    dbutils.fs.rm("/Volumes/rfb/transient/transient/ibge", recurse=True)
else:
    raise Exception("Raw table is empty, please check your source.")